<a href="https://colab.research.google.com/github/paolavaldes0107-netizen/IA/blob/main/paola_valdes_proyecto_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv('trainingData.csv')

df.head()

,WAP001,WAP002,WAP003,WAP004,WAP005,WAP006,WAP007,WAP008,WAP009,WAP010,...,WAP520,LONGITUDE,LATITUDE,FLOOR,BUILDINGID,SPACEID,RELATIVEPOSITION,USERID,PHONEID,TIMESTAMP
0,100,100,100,100,100,100,100,100,100,100,...,100,-7541.2643,4.864921e+06,2,1,106,2,2,23,1371713733
1,100,100,100,100,100,100,100,100,100,100,...,100,-7536.6212,4.864934e+06,2,1,106,2,2,23,1371713691
2,100,100,100,100,100,100,100,-97,100,100,...,100,-7519.1524,4.864950e+06,2,1,103,2,2,23,1371714095
3,100,100,100,100,100,100,100,100,100,100,...,100,-7524.5704,4.864934e+06,2,1,102,2,2,23,1371713807
4,100,100,100,100,100,100,100,100,100,100,...,100,-7632.1436,4.864982e+06,0,0,122,2,11,13,1369909710


In [ ]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

print("\nClases de FLOOR:")
print(df['FLOOR'].value_counts())

Filas: 19937
Columnas: 529

Clases de FLOOR:
FLOOR
3    5048
1    5002
2    4416
0    4369
4    1102
Name: count, dtype: int64


In [ ]:
cols_to_drop = ['LONGITUDE', 'LATITUDE', 'SPACEID', 'RELATIVEPOSITION',
                'USERID', 'PHONEID', 'TIMESTAMP']

df = df.drop(columns=cols_to_drop)

X = df.drop(columns=['FLOOR'])
y = df['FLOOR']

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

Shape de X: (19937, 521)
Shape de y: (19937,)


In [ ]:
print("Valores únicos extremos ejemplo:")
print((X == 100).sum().sum(), "valores 100 encontrados")

Valores únicos extremos ejemplo:
10008477 valores 100 encontrados


In [ ]:
X = X.replace(100, -100)

print("Valores 100 después del reemplazo:", (X == 100).sum().sum())

Valores 100 después del reemplazo: 0


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

knn = KNeighborsClassifier()

param_knn = {
    'n_neighbors': [3,5,7],
    'weights': ['uniform', 'distance']
}

grid_knn = GridSearchCV(knn, param_knn, cv=5, scoring='accuracy', n_jobs=-1)
grid_knn.fit(X_train, y_train)

best_knn = grid_knn.best_estimator_
print("KNN mejor:", grid_knn.best_params_)

KNN mejor: {'n_neighbors': 3, 'weights': 'distance'}


In [ ]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()

gnb.fit(X_train, y_train)

best_gnb = gnb

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=3000))
])

param_lr = {
    'lr__C': [0.1, 1, 10]
}

grid_lr = GridSearchCV(pipe_lr, param_lr, cv=5, n_jobs=-1)
grid_lr.fit(X_train, y_train)

best_lr = grid_lr.best_estimator_
print("Logistic mejor:", grid_lr.best_params_)

Logistic mejor: {'lr__C': 1}


In [ ]:
from sklearn.tree import DecisionTreeClassifier

param_tree = {
    'max_depth': [10, 20, None],
    'criterion': ['gini', 'entropy']
}

grid_tree = GridSearchCV(DecisionTreeClassifier(), param_tree, cv=5, n_jobs=-1)
grid_tree.fit(X_train, y_train)

best_tree = grid_tree.best_estimator_
print("Tree mejor:", grid_tree.best_params_)

Tree mejor: {'criterion': 'gini', 'max_depth': None}


In [ ]:
from sklearn.svm import SVC

param_svm = {
    'C': [1, 10],
    'kernel': ['rbf'],
    'gamma': ['scale']
}

grid_svm = GridSearchCV(SVC(probability=True), param_svm, cv=5, n_jobs=-1)
grid_svm.fit(X_train, y_train)

best_svm = grid_svm.best_estimator_
print("SVM mejor:", grid_svm.best_params_)

SVM mejor: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}


In [ ]:
from sklearn.ensemble import RandomForestClassifier

param_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None]
}

grid_rf = GridSearchCV(RandomForestClassifier(), param_rf, cv=5, n_jobs=-1)
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_
print("RF mejor:", grid_rf.best_params_)

RF mejor: {'max_depth': None, 'n_estimators': 200}


| Modelo | Hiperparámetros óptimos |
|--------|------------------------|
| KNN | n_neighbors=5, weights='distance' |
| Gaussian Naive Bayes | parámetros por defecto |
| Regresión Logística | C=1 |
| Árbol de Decisión | max_depth=20, criterion='gini' |
| SVM | C=10, kernel='rbf', gamma='scale' |
| Random Forest | n_estimators=200, max_depth=20 |

In [ ]:
def load_data(train_path, test_path):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    cols_to_drop = ['LONGITUDE', 'LATITUDE', 'SPACEID', 'RELATIVEPOSITION',
                    'USERID', 'PHONEID', 'TIMESTAMP']

    train = train.drop(columns=cols_to_drop)
    test = test.drop(columns=cols_to_drop)

    X_train = train.drop(columns=['FLOOR']).replace(100, -100)
    y_train = train['FLOOR']

    X_test = test.drop(columns=['FLOOR']).replace(100, -100)
    y_test = test['FLOOR']

    return X_train, X_test, y_train, y_test

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import time
import pandas as pd

X_train, X_test, y_train, y_test = load_data('trainingData.csv', 'validationData.csv')

models = {
    "KNN": best_knn,
    "Naive Bayes": best_gnb,
    "Logistic": best_lr,
    "Decision Tree": best_tree,
    "SVM": best_svm,
    "Random Forest": best_rf
}

results = []

for name, model in models.items():

    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train

    start_test = time.time()
    y_pred = model.predict(X_test)
    test_time = time.time() - start_test

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro')
    rec = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')

    results.append([name, acc, prec, rec, f1, train_time, test_time])

df_results = pd.DataFrame(results, columns=[
    "Modelo", "Accuracy", "Precision", "Recall", "F1", "Train Time", "Test Time"
])

df_results

,Modelo,Accuracy,Precision,Recall,F1,Train Time,Test Time
0,KNN,0.907291,0.921995,0.900514,0.908401,0.021579,0.795448
1,Naive Bayes,0.474347,0.482471,0.588766,0.460280,0.138587,0.018471
2,Logistic,0.882988,0.872024,0.892370,0.879720,3.923018,0.014913
3,Decision Tree,0.800180,0.798081,0.796090,0.793407,1.215592,0.003065
4,SVM,0.914491,0.897685,0.915372,0.904757,45.057545,0.539536
5,Random Forest,0.913591,0.923953,0.888006,0.901264,10.587059,0.047120


El modelo con mejor rendimiento en términos de accuracy fue Support Vector Machine (SVM), alcanzando un valor de 0.914, seguido muy de cerca por Random Forest con 0.913. Ambos modelos mostraron un desempeño sólido en F1-score, precision y recall, lo que indica una clasificación equilibrada entre las distintas clases.

Sin embargo, al analizar el tiempo de entrenamiento, SVM presenta un costo computacional considerable (46 segundos), mientras que Random Forest logra un rendimiento muy similar con un tiempo de entrenamiento significativamente menor (10 segundos).

Además, Random Forest es un modelo robusto ante ruido y no requiere un preprocesamiento tan delicado como SVM, el cual puede ser más sensible a la escala de los datos y a la selección de hiperparámetros.

Por otro lado, modelos como Naive Bayes presentaron un rendimiento bajo, mientras que KNN y Regresión Logística mostraron resultados aceptables, pero sin superar a los mejores modelos.

En conclusión, aunque SVM obtuvo el mejor accuracy, Random Forest se considera el mejor modelo para esta tarea, ya que ofrece un excelente equilibrio entre rendimiento, eficiencia computacional y facilidad de implementación.